# Module 2: Pattern Detection Engine

This notebook implements the pattern-extraction stage of the final project. It takes the structured performance summaries produced by Module 1 and converts them into deterministic, reproducible evidence for later LLM-assisted interpretation.

**Inputs:**  
- `module1_outputs/module1_performance_summary.csv`  
- `module1_outputs/module1_metadata_summary.json`

**Processing steps:**  
1. Validate whether the expected split settings are complete.  
2. Build error curves across batch-aware split ratios.  
3. Compute curve-level numeric features.  
4. Assign rule-based pattern labels.  
5. Generate factual pattern sentences and scenario-level summaries.  
6. Export LLM-ready JSON files and diagnostic visualizations.

**Outputs:**  
All generated CSV, JSON, and PNG files are saved under:

`module2_outputs/`

This module supports the final project repository requirements by making the pattern-detection process documented, reproducible, and easy to inspect.


## Setup and Imports

Input: Python/Colab runtime.

Processing: Import standard libraries and install/import `pandas` if needed.

Output: Required modules are available for later cells.


In [ ]:
# ### Environment and dependency setup
# This cell imports core libraries and defines a reproducible package helper used by later Module 2 steps.

from pathlib import Path
import importlib
import importlib.util
import json
import subprocess
import sys

LOCAL_DEPS = Path.cwd() / ".pydeps_module2"
if LOCAL_DEPS.exists():
    sys.path.insert(0, str(LOCAL_DEPS))


# ### Function: ensure_package
# Install or import a required package and verify that it exposes expected attributes.
def ensure_package(
    import_name: str,
    pip_name: str | None = None,
    required_attrs: tuple[str, ...] = (),
):
    """Install and import a package required by this notebook.

    Inputs:
        import_name: Module name used by Python imports.
        pip_name: Optional pip package name when it differs from import_name.
        required_attrs: Attributes that must exist on the imported module.

    Output:
        Imported module object.
    """
    package_name = pip_name or import_name
    needs_install = importlib.util.find_spec(import_name) is None

    if not needs_install:
        module = importlib.import_module(import_name)
        needs_install = any(not hasattr(module, attr) for attr in required_attrs)
    else:
        module = None

    if needs_install:
        print(f"Installing missing package locally: {package_name}")
        LOCAL_DEPS.mkdir(exist_ok=True)
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--upgrade",
                "--target",
                str(LOCAL_DEPS),
                package_name,
            ]
        )
        sys.path.insert(0, str(LOCAL_DEPS))
        importlib.invalidate_caches()
        sys.modules.pop(import_name, None)
        module = importlib.import_module(import_name)

    missing_attrs = [attr for attr in required_attrs if not hasattr(module, attr)]
    if missing_attrs:
        raise ImportError(
            f"Package {import_name!r} imported but is missing required attributes: {missing_attrs}"
        )

    return importlib.import_module(import_name)


pd = ensure_package("pandas", required_attrs=("read_csv",))


## Constants

Input: Module 1 output naming conventions and required schema.

Processing: Define paths, file names, required columns, and optional columns.

Output: Constants used by validation functions.


In [ ]:
# ### Module 2 paths and schema constants
# These constants point to Module 1 outputs, define Module 2 output names, and specify required input columns.

DEFAULT_MODULE1_DIR = Path("./module1_outputs")
MODULE2_OUTPUT_DIR = Path("./module2_outputs")
PERFORMANCE_FILE = "module1_performance_summary.csv"
METADATA_FILE = "module1_metadata_summary.json"

REQUIRED_COLUMNS = ["scenario", "classifier", "normalization", "split", "error_mean"]
OPTIONAL_COLUMNS = ["batch_balance", "error_sd", "error_min", "error_max", "n_runs"]


## Function Definitions

Input: Module 1 output paths and table schemas.

Processing: Define helpers to resolve paths, load files, validate columns, and prepare input types.

Output: Reusable functions for Module 2 Prompt 1 loading and validation.


In [ ]:
# ### Prompt 1 loading and validation functions
# These helpers resolve Module 1 files, load metadata/CSV tables, validate required columns, and prepare data types.

# ### Function: resolve_module1_paths
# Resolve Module 1 summary and metadata paths and raise clear errors if files are missing.
def resolve_module1_paths(module1_dir: str | Path) -> tuple[Path, Path]:
    """Resolve and validate Module 1 input file paths.

    Inputs:
        module1_dir: Folder expected to contain Module 1 output files.

    Output:
        Tuple containing performance summary path and metadata JSON path.
    """
    module1_path = Path(module1_dir)
    performance_path = module1_path / PERFORMANCE_FILE
    metadata_path = module1_path / METADATA_FILE

    if not performance_path.exists():
        raise FileNotFoundError(f"Missing required Module 1 file: {performance_path}")
    if not metadata_path.exists():
        raise FileNotFoundError(f"Missing required Module 1 file: {metadata_path}")

    return performance_path, metadata_path


# ### Function: load_metadata
# Read Module 1 metadata JSON for use in later audit and JSON outputs.
def load_metadata(metadata_path: str | Path) -> dict:
    """Load Module 1 metadata JSON.

    Inputs:
        metadata_path: Path to module1_metadata_summary.json.

    Output:
        Metadata dictionary loaded from JSON.
    """
    path = Path(metadata_path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required Module 1 file: {path}")

    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


# ### Function: load_performance_summary
# Load the Module 1 performance CSV into a DataFrame.
def load_performance_summary(performance_path: str | Path):
    """Load Module 1 performance summary CSV.

    Inputs:
        performance_path: Path to module1_performance_summary.csv.

    Output:
        DataFrame containing Module 1 performance summary rows.
    """
    path = Path(performance_path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required Module 1 file: {path}")

    return pd.read_csv(path)


# ### Function: validate_performance_columns
# Check that required columns and optional expected columns are present in the performance table.
def validate_performance_columns(performance_df) -> None:
    """Validate required and optional performance summary columns.

    Inputs:
        performance_df: DataFrame loaded from Module 1 performance summary.

    Output:
        None. Raises ValueError for missing required columns and warns for selected optional columns.
    """
    missing_required = [column for column in REQUIRED_COLUMNS if column not in performance_df.columns]
    if missing_required:
        raise ValueError(f"Performance summary is missing required columns: {missing_required}")

    for optional_column in ["error_sd", "n_runs"]:
        if optional_column not in performance_df.columns:
            print(f"Warning: optional column '{optional_column}' is missing; continuing without it.")


# ### Function: prepare_performance_df
# Convert split and numeric fields to analysis-ready data types.
def prepare_performance_df(performance_df):
    """Prepare Module 1 performance summary for later Module 2 steps.

    Inputs:
        performance_df: DataFrame loaded from Module 1 performance summary.

    Output:
        Validated copy of performance_df with integer split and batch_balance fallback.
    """
    prepared_df = performance_df.copy()

    if "batch_balance" not in prepared_df.columns:
        prepared_df["batch_balance"] = "unknown"

    validate_performance_columns(prepared_df)

    prepared_df["split"] = pd.to_numeric(prepared_df["split"], errors="raise").astype(int)
    return prepared_df


# ### Function: load_and_validate_module1_outputs
# Run the complete Prompt 1 loading, validation, type preparation, and output-directory setup.
def load_and_validate_module1_outputs(module1_dir: str | Path, output_dir: str | Path):
    """Load Module 1 outputs and validate the inputs needed for Module 2 Prompt 1.

    Inputs:
        module1_dir: Folder containing Module 1 performance summary and metadata files.
        output_dir: Folder to create for future Module 2 outputs.

    Output:
        Tuple of prepared performance DataFrame and metadata dictionary.
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    performance_path, metadata_path = resolve_module1_paths(module1_dir)
    performance_df = load_performance_summary(performance_path)
    metadata = load_metadata(metadata_path)
    performance_df = prepare_performance_df(performance_df)

    return performance_df, metadata


## Run Module 2 Prompt 1 Loading

Input: `./module1_outputs` containing Module 1 performance summary and metadata.

Processing: Create `./module2_outputs`, load Module 1 outputs, validate required fields, and normalize `split`.

Output: `performance_df` and `metadata` objects ready for later Module 2 steps.


In [ ]:
# ### Execute Prompt 1
# Load and validate Module 1 outputs before any pattern detection logic is applied.

module1_dir = Path("./module1_outputs")
output_dir = Path("./module2_outputs")

performance_df, metadata = load_and_validate_module1_outputs(module1_dir, output_dir)


## Display Validation Output

Input: Loaded `performance_df` and `metadata`.

Processing: Print metadata, preview the performance table, and display its shape.

Output: Human-readable confirmation that Prompt 1 completed successfully.


In [ ]:
# ### Display Prompt 1 validation results
# Preview metadata and the performance summary so the input state is transparent.

print("Metadata:")
print(json.dumps(metadata, indent=2))

print("Performance summary preview:")
display(performance_df.head())

print("Performance summary shape:")
display(performance_df.shape)

print("Prompt 1 completed: Module 1 outputs loaded and validated.")


## Prompt 2 Split Completeness Functions

Input: `performance_df` from Prompt 1 and the Module 2 output directory.

Processing: Define expected split values, grouping keys, and a function that checks whether each error curve has all required splits.

Output: Reusable split-completeness checking function for Prompt 2.


In [ ]:
# ### Prompt 2 split-completeness functions
# These functions verify that each experimental curve contains all expected split points.

EXPECTED_SPLITS = [50, 70, 80, 90, 100]
COMPLETENESS_KEYS = ["scenario", "batch_balance", "classifier", "normalization"]
MISSING_SPLIT_REPORT_FILE = "missing_split_report.csv"
MISSING_SPLIT_REPORT_COLUMNS = COMPLETENESS_KEYS + ["observed_splits", "missing_splits"]


# ### Function: check_split_completeness
# Identify complete and incomplete error curves based on required split values and save a missing-split report.
def check_split_completeness(performance_df, output_dir: str | Path):
    """Check whether every error curve contains all expected split values.

    Inputs:
        performance_df: Validated Module 1 performance summary DataFrame from Prompt 1.
        output_dir: Directory where missing_split_report.csv should be written.

    Output:
        Tuple of complete_df and missing_report_df. complete_df contains only complete curves.
    """
    missing_key_columns = [column for column in COMPLETENESS_KEYS if column not in performance_df.columns]
    if missing_key_columns:
        raise ValueError(f"performance_df is missing completeness key columns: {missing_key_columns}")
    if "split" not in performance_df.columns:
        raise ValueError("performance_df is missing required column: split")

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    check_df = performance_df.copy()
    check_df["split"] = pd.to_numeric(check_df["split"], errors="raise").astype(int)

    complete_keys = []
    missing_rows = []

    for key_values, group_df in check_df.groupby(COMPLETENESS_KEYS, dropna=False):
        if not isinstance(key_values, tuple):
            key_values = (key_values,)
        key_dict = dict(zip(COMPLETENESS_KEYS, key_values))
        observed_splits = sorted(int(value) for value in group_df["split"].unique().tolist())
        missing_splits = [split for split in EXPECTED_SPLITS if split not in observed_splits]

        if missing_splits:
            missing_rows.append(
                {
                    **key_dict,
                    "observed_splits": ",".join(str(value) for value in observed_splits),
                    "missing_splits": ",".join(str(value) for value in missing_splits),
                }
            )
        else:
            complete_keys.append(key_values)

    if complete_keys:
        complete_key_index = pd.MultiIndex.from_tuples(complete_keys, names=COMPLETENESS_KEYS)
        row_key_index = pd.MultiIndex.from_frame(check_df[COMPLETENESS_KEYS])
        complete_df = check_df[row_key_index.isin(complete_key_index)].copy().reset_index(drop=True)
    else:
        complete_df = check_df.iloc[0:0].copy()

    missing_report_df = pd.DataFrame(missing_rows, columns=MISSING_SPLIT_REPORT_COLUMNS)
    missing_report_path = output_path / MISSING_SPLIT_REPORT_FILE
    missing_report_df.to_csv(missing_report_path, index=False)

    if complete_df.empty:
        raise ValueError("No complete error curve found after split completeness check.")

    return complete_df, missing_report_df


## Run Prompt 2 Split Completeness Check

Input: `performance_df` loaded in Prompt 1.

Processing: Check complete split coverage for each `scenario + batch_balance + classifier + normalization` combination, save the missing split report, and keep only complete curves in `complete_df`.

Output: `complete_df`, `missing_report_df`, `module2_outputs/missing_split_report.csv`, and a printed Prompt 2 status summary.


In [ ]:
# ### Execute Prompt 2
# Keep only complete curves and write a missing-split report for reproducibility.

complete_df, missing_report_df = check_split_completeness(performance_df, output_dir)

original_combination_count = performance_df[COMPLETENESS_KEYS].drop_duplicates().shape[0]
complete_combination_count = complete_df[COMPLETENESS_KEYS].drop_duplicates().shape[0]
missing_combination_count = missing_report_df.shape[0]

print(f"Original combination count: {original_combination_count}")
print(f"Complete combination count: {complete_combination_count}")
print(f"Missing combination count: {missing_combination_count}")

if not missing_report_df.empty:
    print("Missing split report:")
    display(missing_report_df)

print("Complete data preview:")
display(complete_df.head())

print("Prompt 2 completed: split completeness checked.")


## Prompt 3 Error Curve Table Functions

Input: `complete_df` from Prompt 2.

Processing: Import `numpy`, define curve-table constants, and define a function that pivots each complete long-format error curve into one wide row.

Output: Reusable function for generating `module2_error_curve_table.csv`.


In [ ]:
# ### Prompt 3 error-curve table functions
# These functions pivot complete long-format records into one wide row per scenario/classifier/normalization curve.

np = ensure_package("numpy")

CURVE_KEYS = ["scenario", "batch_balance", "classifier", "normalization"]
ERROR_CURVE_FILE = "module2_error_curve_table.csv"
ERROR_COLUMNS = ["error_50", "error_70", "error_80", "error_90", "error_100"]
SD_COLUMNS = ["sd_50", "sd_70", "sd_80", "sd_90", "sd_100"]
N_RUNS_COLUMNS = ["n_runs_50", "n_runs_70", "n_runs_80", "n_runs_90", "n_runs_100"]
CURVE_COLUMNS = CURVE_KEYS + ERROR_COLUMNS + SD_COLUMNS + N_RUNS_COLUMNS


# ### Function: build_error_curve_table
# Pivot complete long-format performance summaries into one wide row per error curve.
def build_error_curve_table(complete_df, output_dir: str | Path):
    """Convert complete long-format performance rows into wide error-curve rows.

    Inputs:
        complete_df: Complete split-validated performance DataFrame from Prompt 2.
        output_dir: Directory where module2_error_curve_table.csv should be saved.

    Output:
        curve_df with one row per scenario, batch balance, classifier, and normalization.
    """
    missing_key_columns = [column for column in CURVE_KEYS if column not in complete_df.columns]
    if missing_key_columns:
        raise ValueError(f"complete_df is missing curve key columns: {missing_key_columns}")
    for required_column in ["split", "error_mean"]:
        if required_column not in complete_df.columns:
            raise ValueError(f"complete_df is missing required column: {required_column}")

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    curve_source = complete_df.copy()
    curve_source["split"] = pd.to_numeric(curve_source["split"], errors="raise").astype(int)

    if "error_sd" not in curve_source.columns:
        curve_source["error_sd"] = np.nan
    if "n_runs" not in curve_source.columns:
        curve_source["n_runs"] = np.nan

    curve_rows = []
    for key_values, group_df in curve_source.groupby(CURVE_KEYS, dropna=False):
        if not isinstance(key_values, tuple):
            key_values = (key_values,)
        row = dict(zip(CURVE_KEYS, key_values))

        split_indexed = group_df.set_index("split")
        for split in EXPECTED_SPLITS:
            if split not in split_indexed.index:
                row[f"error_{split}"] = np.nan
                row[f"sd_{split}"] = np.nan
                row[f"n_runs_{split}"] = np.nan
                continue

            split_row = split_indexed.loc[split]
            if isinstance(split_row, pd.DataFrame):
                split_row = split_row.iloc[0]
            row[f"error_{split}"] = split_row["error_mean"]
            row[f"sd_{split}"] = split_row["error_sd"]
            row[f"n_runs_{split}"] = split_row["n_runs"]

        curve_rows.append(row)

    curve_df = pd.DataFrame(curve_rows, columns=CURVE_COLUMNS)
    curve_df = curve_df.sort_values(CURVE_KEYS).reset_index(drop=True)

    curve_output_path = output_path / ERROR_CURVE_FILE
    try:
        curve_df.to_csv(curve_output_path, index=False)
    except PermissionError:
        if curve_output_path.exists():
            print(
                f"Warning: could not overwrite locked file {curve_output_path}; "
                "continuing with in-memory curve_df."
            )
        else:
            raise

    return curve_df


## Run Prompt 3 Error Curve Table Generation

Input: `complete_df` from Prompt 2.

Processing: Generate a wide curve-format table, save it to `module2_outputs/module2_error_curve_table.csv`, and check whether any `error_*` columns contain missing values.

Output: `curve_df`, saved curve table CSV, preview, shape, and Prompt 3 completion message.


In [ ]:
# ### Execute Prompt 3
# Build and save the wide error-curve table for downstream feature engineering.

curve_df = build_error_curve_table(complete_df, output_dir)

print("Error curve table preview:")
display(curve_df.head())

print("Error curve table shape:")
display(curve_df.shape)

missing_error_rows = curve_df[curve_df[ERROR_COLUMNS].isna().any(axis=1)]
if not missing_error_rows.empty:
    print("Warning: NA values found in error curve columns. Affected rows:")
    display(missing_error_rows)

print("Prompt 3 completed: error curve table generated.")


## Prompt 4 Numeric Feature Functions

Input: `curve_df` from Prompt 3.

Processing: Define numeric feature constants and functions that compute curve-level numeric features from the five split errors.

Output: Reusable function for generating `module2_numeric_features.csv`.


In [ ]:
# ### Prompt 4 numeric feature functions
# These functions turn each five-point error curve into interpretable numeric features such as slope, max jump, and total degradation.

NUMERIC_FEATURES_FILE = "module2_numeric_features.csv"
SPLIT_VALUES = [50, 70, 80, 90, 100]
STEP_INTERVALS = ["50-70", "70-80", "80-90", "90-100"]
STEP_COLUMNS = ["step_50_70", "step_70_80", "step_80_90", "step_90_100"]
NUMERIC_FEATURE_COLUMNS = (
    CURVE_KEYS
    + ERROR_COLUMNS
    + [
        "delta_100_50",
        "relative_increase_pct",
        "mean_error",
        "sd_across_splits",
        "range_error",
    ]
    + STEP_COLUMNS
    + [
        "max_jump",
        "max_jump_interval",
        "min_jump",
        "largest_drop_interval",
        "abs_max_step_change",
        "abs_max_step_interval",
        "slope",
        "monotonic_increasing",
        "monotonic_decreasing",
    ]
)


# ### Helper: _linear_slope
# Compute a simple linear slope for error as a function of split value.
def _linear_slope(errors) -> float:
    """Compute linear regression slope for errors over SPLIT_VALUES.

    Inputs:
        errors: Ordered iterable of error values for splits 50, 70, 80, 90, and 100.

    Output:
        Floating-point slope from a first-degree polynomial fit.
    """
    return float(np.polyfit(np.array(SPLIT_VALUES, dtype=float), np.array(errors, dtype=float), 1)[0])


# ### Helper: _interval_for_value
# Map an observed step value back to the split interval where it occurs.
def _interval_for_value(values, intervals, mode: str) -> str:
    """Return the interval associated with an extreme step-change value.

    Inputs:
        values: Ordered numeric step changes.
        intervals: Interval labels aligned with values.
        mode: One of 'max', 'min', or 'absmax'.

    Output:
        Interval label corresponding to the requested extreme.
    """
    value_array = np.array(values, dtype=float)
    if mode == "max":
        index = int(np.nanargmax(value_array))
    elif mode == "min":
        index = int(np.nanargmin(value_array))
    elif mode == "absmax":
        index = int(np.nanargmax(np.abs(value_array)))
    else:
        raise ValueError(f"Unsupported interval selection mode: {mode}")
    return intervals[index]


# ### Function: build_numeric_features
# Compute degradation, variability, slope, and step-change features from each error curve.
def build_numeric_features(curve_df, output_dir: str | Path):
    """Compute numeric pattern features from wide error curves.

    Inputs:
        curve_df: Wide error curve DataFrame generated by Prompt 3.
        output_dir: Directory where module2_numeric_features.csv should be saved.

    Output:
        numeric_features_df with one row per curve and numeric feature columns.
    """
    missing_columns = [column for column in CURVE_KEYS + ERROR_COLUMNS if column not in curve_df.columns]
    if missing_columns:
        raise ValueError(f"curve_df is missing required columns: {missing_columns}")

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    feature_df = curve_df.copy()
    for column in ERROR_COLUMNS:
        feature_df[column] = pd.to_numeric(feature_df[column], errors="coerce")

    out_of_range_rows = feature_df[
        ((feature_df[ERROR_COLUMNS] < 0) | (feature_df[ERROR_COLUMNS] > 1)).any(axis=1)
    ]
    if not out_of_range_rows.empty:
        print("Warning: error values outside [0, 1] found in these rows:")
        display(out_of_range_rows)

    errors = feature_df[ERROR_COLUMNS]
    feature_df["delta_100_50"] = feature_df["error_100"] - feature_df["error_50"]
    feature_df["relative_increase_pct"] = np.where(
        feature_df["error_50"] == 0,
        np.nan,
        feature_df["delta_100_50"] / feature_df["error_50"] * 100,
    )
    feature_df["mean_error"] = errors.mean(axis=1)
    feature_df["sd_across_splits"] = errors.std(axis=1)
    feature_df["range_error"] = errors.max(axis=1) - errors.min(axis=1)

    feature_df["step_50_70"] = feature_df["error_70"] - feature_df["error_50"]
    feature_df["step_70_80"] = feature_df["error_80"] - feature_df["error_70"]
    feature_df["step_80_90"] = feature_df["error_90"] - feature_df["error_80"]
    feature_df["step_90_100"] = feature_df["error_100"] - feature_df["error_90"]

    steps = feature_df[STEP_COLUMNS]
    feature_df["max_jump"] = steps.max(axis=1)
    feature_df["max_jump_interval"] = steps.apply(
        lambda row: _interval_for_value(row.tolist(), STEP_INTERVALS, "max"), axis=1
    )
    feature_df["min_jump"] = steps.min(axis=1)
    feature_df["largest_drop_interval"] = steps.apply(
        lambda row: _interval_for_value(row.tolist(), STEP_INTERVALS, "min"), axis=1
    )
    feature_df["abs_max_step_change"] = steps.abs().max(axis=1)
    feature_df["abs_max_step_interval"] = steps.apply(
        lambda row: _interval_for_value(row.tolist(), STEP_INTERVALS, "absmax"), axis=1
    )

    feature_df["slope"] = errors.apply(lambda row: _linear_slope(row.tolist()), axis=1)
    feature_df["monotonic_increasing"] = steps.ge(-1e-8).all(axis=1)
    feature_df["monotonic_decreasing"] = steps.le(1e-8).all(axis=1)

    numeric_features_df = feature_df[list(NUMERIC_FEATURE_COLUMNS)].copy()
    if len(numeric_features_df) != len(curve_df):
        raise ValueError(
            f"numeric_features_df row count {len(numeric_features_df)} does not match curve_df row count {len(curve_df)}."
        )

    numeric_output_path = output_path / NUMERIC_FEATURES_FILE
    try:
        numeric_features_df.to_csv(numeric_output_path, index=False)
    except PermissionError:
        if numeric_output_path.exists():
            print(
                f"Warning: could not overwrite locked file {numeric_output_path}; "
                "continuing with in-memory numeric_features_df."
            )
        else:
            raise

    return numeric_features_df


## Run Prompt 4 Numeric Feature Generation

Input: `curve_df` from Prompt 3.

Processing: Compute numeric curve features, save them to `module2_outputs/module2_numeric_features.csv`, and print feature diagnostics.

Output: `numeric_features_df`, saved numeric features CSV, preview, shape, and Prompt 4 completion message.


In [ ]:
# ### Execute Prompt 4
# Compute numeric features and print diagnostics for the cross-batch degradation signal.

numeric_features_df = build_numeric_features(curve_df, output_dir)

print("Numeric features preview:")
display(numeric_features_df.head())

print("Numeric features shape:")
display(numeric_features_df.shape)

print("delta_100_50 summary:")
print(numeric_features_df["delta_100_50"].describe())

print("Prompt 4 completed: numeric pattern features generated.")


## Prompt 5 Rule-Based Pattern Label Functions

Input: `numeric_features_df` from Prompt 4.

Processing: Define label columns and deterministic rule functions that add rule-based pattern labels to numeric features.

Output: Reusable function for generating `module2_pattern_table.csv`.


In [ ]:
# ### Prompt 5 deterministic pattern-label functions
# These rule-based functions translate numeric features into interpretable labels without using an LLM.

PATTERN_TABLE_FILE = "module2_pattern_table.csv"
LABEL_COLUMNS = [
    "trend_label",
    "robustness_flag",
    "spike_flag",
    "spike_type",
    "curve_shape",
    "pattern_strength",
    "degradation_type",
]


# ### Helper: _as_bool
# Normalize boolean-like values for robust conditional logic.
def _as_bool(value) -> bool:
    """Convert common boolean-like values to a Python bool.

    Inputs:
        value: Boolean or boolean-like scalar value.

    Output:
        Boolean interpretation of the input value.
    """
    if isinstance(value, str):
        return value.strip().lower() == "true"
    return bool(value)


# ### Helper: _label_trend
# Assign an overall trend label from curve-level numeric features.
def _label_trend(row) -> str:
    """Assign trend_label using slope and total error change.

    Inputs:
        row: One numeric feature row.

    Output:
        trend_label string.
    """
    if row["slope"] > 0.001 and row["delta_100_50"] > 0.03:
        return "increasing"
    if row["slope"] < -0.001 and row["delta_100_50"] < -0.03:
        return "decreasing"
    return "flat_or_weak"


# ### Helper: _label_robustness
# Flag whether a curve appears robust or degraded under stronger batch separation.
def _label_robustness(delta_100_50: float) -> str:
    """Assign robustness_flag from total error change.

    Inputs:
        delta_100_50: Difference between error_100 and error_50.

    Output:
        robustness_flag string.
    """
    if delta_100_50 <= -0.03:
        return "improved_at_high_split"
    if -0.03 < delta_100_50 < 0.03:
        return "relatively_stable"
    if 0.03 <= delta_100_50 < 0.10:
        return "moderately_sensitive"
    return "batch_sensitive"


# ### Helper: _label_spike_type
# Classify whether the curve has a sharp local jump.
def _label_spike_type(spike_flag: bool, abs_max_step_interval: str) -> str:
    """Assign spike_type from spike flag and largest absolute step interval.

    Inputs:
        spike_flag: Whether the curve has a major step change.
        abs_max_step_interval: Interval with largest absolute step change.

    Output:
        spike_type string.
    """
    if spike_flag and abs_max_step_interval == "90-100":
        return "late_extreme_split_spike"
    if spike_flag:
        return "intermediate_spike"
    return "no_major_spike"


# ### Helper: _label_curve_shape
# Summarize the overall curve shape from step changes.
def _label_curve_shape(row) -> str:
    """Assign curve_shape from monotonicity, spike flag, and trend label.

    Inputs:
        row: One pattern table row containing trend and spike labels.

    Output:
        curve_shape string.
    """
    monotonic_increasing = _as_bool(row["monotonic_increasing"])
    monotonic_decreasing = _as_bool(row["monotonic_decreasing"])
    spike_flag = _as_bool(row["spike_flag"])

    if monotonic_increasing and spike_flag:
        return "monotonic_with_spike"
    if monotonic_increasing and not spike_flag:
        return "gradual_monotonic_increase"
    if monotonic_decreasing:
        return "monotonic_decrease"
    if (not monotonic_increasing) and (not monotonic_decreasing) and row["trend_label"] == "increasing":
        return "non_monotonic_increase"
    if row["trend_label"] == "flat_or_weak":
        return "mostly_flat"
    return "other"


# ### Helper: _label_pattern_strength
# Categorize the magnitude of degradation or stability.
def _label_pattern_strength(delta_100_50: float) -> str:
    """Assign pattern_strength from absolute total error change.

    Inputs:
        delta_100_50: Difference between error_100 and error_50.

    Output:
        pattern_strength string.
    """
    abs_delta = abs(delta_100_50)
    if abs_delta >= 0.20:
        return "strong"
    if 0.10 <= abs_delta < 0.20:
        return "moderate"
    if 0.03 <= abs_delta < 0.10:
        return "weak"
    return "minimal"


# ### Helper: _label_degradation_type
# Name the type of performance degradation represented by the curve.
def _label_degradation_type(row) -> str:
    """Assign degradation_type from total change, spike behavior, and monotonicity.

    Inputs:
        row: One pattern table row containing numeric features and rule labels.

    Output:
        degradation_type string.
    """
    spike_flag = _as_bool(row["spike_flag"])
    monotonic_increasing = _as_bool(row["monotonic_increasing"])
    monotonic_decreasing = _as_bool(row["monotonic_decreasing"])

    if row["delta_100_50"] <= -0.03:
        return "improving"
    if spike_flag and row["abs_max_step_interval"] == "90-100":
        return "late_spike_driven"
    if spike_flag:
        return "intermediate_spike_driven"
    if monotonic_increasing and not spike_flag:
        return "gradual_degradation"
    if (not monotonic_increasing) and (not monotonic_decreasing):
        return "fluctuating"
    return "weak_or_unclear"


# ### Function: build_pattern_table
# Apply deterministic rules to numeric features and save a labeled pattern table.
def build_pattern_table(numeric_features_df, output_dir: str | Path):
    """Add rule-based labels to numeric features and save the pattern table.

    Inputs:
        numeric_features_df: Numeric feature DataFrame from Prompt 4.
        output_dir: Directory where module2_pattern_table.csv should be saved.

    Output:
        pattern_df containing all numeric feature columns plus LABEL_COLUMNS.
    """
    required_columns = [
        "slope",
        "delta_100_50",
        "abs_max_step_change",
        "abs_max_step_interval",
        "monotonic_increasing",
        "monotonic_decreasing",
    ]
    missing_columns = [column for column in required_columns if column not in numeric_features_df.columns]
    if missing_columns:
        raise ValueError(f"numeric_features_df is missing required columns: {missing_columns}")

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    pattern_df = numeric_features_df.copy()
    pattern_df["trend_label"] = pattern_df.apply(_label_trend, axis=1)
    pattern_df["robustness_flag"] = pattern_df["delta_100_50"].apply(_label_robustness)
    pattern_df["spike_flag"] = pattern_df["abs_max_step_change"] >= 0.10
    pattern_df["spike_type"] = pattern_df.apply(
        lambda row: _label_spike_type(_as_bool(row["spike_flag"]), row["abs_max_step_interval"]),
        axis=1,
    )
    pattern_df["curve_shape"] = pattern_df.apply(_label_curve_shape, axis=1)
    pattern_df["pattern_strength"] = pattern_df["delta_100_50"].apply(_label_pattern_strength)
    pattern_df["degradation_type"] = pattern_df.apply(_label_degradation_type, axis=1)

    if len(pattern_df) != len(numeric_features_df):
        raise ValueError(
            f"pattern_df row count {len(pattern_df)} does not match numeric_features_df row count {len(numeric_features_df)}."
        )

    pattern_output_path = output_path / PATTERN_TABLE_FILE
    try:
        pattern_df.to_csv(pattern_output_path, index=False)
    except PermissionError:
        if pattern_output_path.exists():
            print(
                f"Warning: could not overwrite locked file {pattern_output_path}; "
                "continuing with in-memory pattern_df."
            )
        else:
            raise

    return pattern_df


## Run Prompt 5 Rule-Based Label Generation

Input: `numeric_features_df` from Prompt 4.

Processing: Add deterministic rule-based labels, save `module2_pattern_table.csv`, print label distributions, and check label completeness.

Output: `pattern_df`, saved pattern table CSV, label count summaries, and Prompt 5 completion message.


In [ ]:
# ### Execute Prompt 5
# Build the pattern table, save it, and print label distributions for validation.

pattern_df = build_pattern_table(numeric_features_df, output_dir)

print("Pattern table preview:")
display(pattern_df.head())

for label_column in [
    "trend_label",
    "robustness_flag",
    "spike_type",
    "curve_shape",
    "pattern_strength",
    "degradation_type",
]:
    print(f"{label_column} value counts:")
    print(pattern_df[label_column].value_counts(dropna=False))
    print()

missing_label_rows = pattern_df[pattern_df[LABEL_COLUMNS].isna().any(axis=1)]
if not missing_label_rows.empty:
    print("Warning: missing values found in label columns. Affected rows:")
    display(missing_label_rows)

print("Prompt 5 completed: rule-based pattern labels generated.")


## Prompt 6 Pattern Sentence And Classifier Summary Functions

Input: `pattern_df` from Prompt 5.

Processing: Define factual sentence generation and classifier-level aggregation helpers.

Output: Reusable functions for saving sentence-enhanced pattern tables and classifier summaries.


In [ ]:
# ### Prompt 6 pattern-sentence and classifier-summary functions
# These functions create factual natural-language summaries grounded in numeric features and aggregate them by classifier.

PATTERN_SENTENCE_FILE = "module2_pattern_table_with_sentences.csv"
CLASSIFIER_SUMMARY_FILE = "module2_classifier_summary.csv"
PATTERN_SENTENCE_COLUMNS = [
    "scenario",
    "classifier",
    "normalization",
    "error_50",
    "error_100",
    "delta_100_50",
    "trend_label",
    "robustness_flag",
    "pattern_strength",
    "degradation_type",
    "spike_type",
    "abs_max_step_interval",
]
CLASSIFIER_SUMMARY_COLUMNS = [
    "scenario",
    "batch_balance",
    "classifier",
    "n_normalizations",
    "n_batch_sensitive",
    "n_moderately_sensitive",
    "n_relatively_stable",
    "n_improved_at_high_split",
    "n_spike_curves",
    "n_late_spike_driven",
    "n_gradual_degradation",
    "n_fluctuating",
    "mean_delta_100_50",
    "max_delta_100_50",
    "min_delta_100_50",
    "mean_abs_delta_100_50",
    "most_sensitive_normalization",
    "least_sensitive_normalization",
    "dominant_curve_shape",
    "dominant_degradation_type",
    "dominant_robustness_flag",
    "classifier_summary_sentence",
]


# ### Function: build_pattern_sentence
# Create one concise factual sentence describing a curve using observed numeric values and rule labels.
def build_pattern_sentence(row) -> str:
    """Build one factual pattern sentence for a single error curve.

    Inputs:
        row: One row from pattern_df containing curve errors and rule-based labels.

    Output:
        A human-readable sentence describing the observed curve pattern.
    """
    return (
        f"For scenario {row['scenario']}, {row['classifier']} with {row['normalization']} "
        f"normalization shows {row['trend_label']} error from {row['error_50']:.3f} "
        f"at split=50 to {row['error_100']:.3f} at split=100. "
        f"The delta_100_50 is {row['delta_100_50']:.3f}; "
        f"robustness_flag={row['robustness_flag']}, "
        f"pattern_strength={row['pattern_strength']}, "
        f"degradation_type={row['degradation_type']}, and "
        f"spike_type={row['spike_type']} with the largest absolute step change "
        f"at {row['abs_max_step_interval']}."
    )


# ### Helper: _mode_value
# Return the most frequent non-missing label value for summary text.
def _mode_value(series) -> object:
    """Return the most frequent value with deterministic tie-breaking.

    Inputs:
        series: Pandas Series containing categorical values.

    Output:
        Most frequent value; ties are resolved by sorted value order.
    """
    non_missing = series.dropna()
    if non_missing.empty:
        return None
    counts = non_missing.astype(str).value_counts()
    max_count = counts.max()
    candidates = sorted(counts[counts == max_count].index.tolist())
    return candidates[0]


# ### Function: add_pattern_sentences
# Attach pattern sentences to the row-level pattern table and save the result.
def add_pattern_sentences(pattern_df, output_dir: str | Path):
    """Add factual pattern_sentence values and save the sentence-enhanced table.

    Inputs:
        pattern_df: Pattern table generated by Prompt 5.
        output_dir: Directory where module2_pattern_table_with_sentences.csv should be saved.

    Output:
        pattern_sentence_df preserving all pattern_df columns plus pattern_sentence.
    """
    missing_columns = [column for column in PATTERN_SENTENCE_COLUMNS if column not in pattern_df.columns]
    if missing_columns:
        raise ValueError(f"pattern_df is missing sentence input columns: {missing_columns}")

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    pattern_sentence_df = pattern_df.copy()
    pattern_sentence_df["pattern_sentence"] = pattern_sentence_df.apply(build_pattern_sentence, axis=1)

    empty_sentence_rows = pattern_sentence_df[
        pattern_sentence_df["pattern_sentence"].isna()
        | (pattern_sentence_df["pattern_sentence"].astype(str).str.strip() == "")
    ]
    if not empty_sentence_rows.empty:
        print("Warning: empty pattern_sentence values found in these rows:")
        display(empty_sentence_rows)

    sentence_output_path = output_path / PATTERN_SENTENCE_FILE
    try:
        pattern_sentence_df.to_csv(sentence_output_path, index=False)
    except PermissionError:
        if sentence_output_path.exists():
            print(
                f"Warning: could not overwrite locked file {sentence_output_path}; "
                "continuing with in-memory pattern_sentence_df."
            )
        else:
            raise

    return pattern_sentence_df


# ### Function: build_classifier_summary
# Aggregate pattern information across normalizations for each scenario/classifier pair.
def build_classifier_summary(pattern_df, output_dir: str | Path):
    """Aggregate curve-level labels into classifier-level pattern summaries.

    Inputs:
        pattern_df: Pattern table generated by Prompt 5.
        output_dir: Directory where module2_classifier_summary.csv should be saved.

    Output:
        classifier_summary_df with one row per scenario, batch_balance, and classifier.
    """
    required_columns = [
        "scenario",
        "batch_balance",
        "classifier",
        "normalization",
        "delta_100_50",
        "robustness_flag",
        "spike_flag",
        "degradation_type",
        "curve_shape",
    ]
    missing_columns = [column for column in required_columns if column not in pattern_df.columns]
    if missing_columns:
        raise ValueError(f"pattern_df is missing classifier summary columns: {missing_columns}")

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    summary_rows = []
    group_keys = ["scenario", "batch_balance", "classifier"]
    for key_values, group_df in pattern_df.groupby(group_keys, dropna=False):
        if not isinstance(key_values, tuple):
            key_values = (key_values,)
        row = dict(zip(group_keys, key_values))

        delta_values = pd.to_numeric(group_df["delta_100_50"], errors="coerce")
        max_delta_index = delta_values.idxmax()
        min_delta_index = delta_values.idxmin()
        spike_values = group_df["spike_flag"].apply(_as_bool)

        row["n_normalizations"] = int(group_df["normalization"].nunique())
        row["n_batch_sensitive"] = int((group_df["robustness_flag"] == "batch_sensitive").sum())
        row["n_moderately_sensitive"] = int((group_df["robustness_flag"] == "moderately_sensitive").sum())
        row["n_relatively_stable"] = int((group_df["robustness_flag"] == "relatively_stable").sum())
        row["n_improved_at_high_split"] = int((group_df["robustness_flag"] == "improved_at_high_split").sum())
        row["n_spike_curves"] = int(spike_values.sum())
        row["n_late_spike_driven"] = int((group_df["degradation_type"] == "late_spike_driven").sum())
        row["n_gradual_degradation"] = int((group_df["degradation_type"] == "gradual_degradation").sum())
        row["n_fluctuating"] = int((group_df["degradation_type"] == "fluctuating").sum())
        row["mean_delta_100_50"] = float(delta_values.mean())
        row["max_delta_100_50"] = float(delta_values.max())
        row["min_delta_100_50"] = float(delta_values.min())
        row["mean_abs_delta_100_50"] = float(delta_values.abs().mean())
        row["most_sensitive_normalization"] = group_df.loc[max_delta_index, "normalization"]
        row["least_sensitive_normalization"] = group_df.loc[min_delta_index, "normalization"]
        row["dominant_curve_shape"] = _mode_value(group_df["curve_shape"])
        row["dominant_degradation_type"] = _mode_value(group_df["degradation_type"])
        row["dominant_robustness_flag"] = _mode_value(group_df["robustness_flag"])
        row["classifier_summary_sentence"] = (
            f"For classifier {row['classifier']}, {row['n_batch_sensitive']} out of "
            f"{row['n_normalizations']} normalization settings are batch-sensitive. "
            f"The most sensitive normalization is {row['most_sensitive_normalization']}, "
            f"and the least sensitive normalization is {row['least_sensitive_normalization']}. "
            f"The dominant degradation type is {row['dominant_degradation_type']}."
        )
        summary_rows.append(row)

    classifier_summary_df = pd.DataFrame(summary_rows, columns=CLASSIFIER_SUMMARY_COLUMNS)
    classifier_summary_df = classifier_summary_df.sort_values(
        ["scenario", "batch_balance", "classifier"]
    ).reset_index(drop=True)

    summary_output_path = output_path / CLASSIFIER_SUMMARY_FILE
    try:
        classifier_summary_df.to_csv(summary_output_path, index=False)
    except PermissionError:
        if summary_output_path.exists():
            print(
                f"Warning: could not overwrite locked file {summary_output_path}; "
                "continuing with in-memory classifier_summary_df."
            )
        else:
            raise

    return classifier_summary_df


## Run Prompt 6 Pattern Sentences And Classifier Summary

Input: `pattern_df` from Prompt 5.

Processing: Generate pattern sentences, build classifier-level summaries, save both CSV outputs, and validate row counts/files.

Output: `pattern_sentence_df`, `classifier_summary_df`, saved CSV files, previews, and Prompt 6 completion message.


In [ ]:
# ### Execute Prompt 6
# Generate pattern sentences and classifier-level summaries used by the LLM-ready output.

pattern_sentence_df = add_pattern_sentences(pattern_df, output_dir)
pattern_df = pattern_sentence_df
classifier_summary_df = build_classifier_summary(pattern_df, output_dir)

print("Pattern sentence preview:")
display(pattern_sentence_df[["scenario", "classifier", "normalization", "pattern_sentence"]].head())

empty_sentence_rows = pattern_sentence_df[
    pattern_sentence_df["pattern_sentence"].isna()
    | (pattern_sentence_df["pattern_sentence"].astype(str).str.strip() == "")
]
if not empty_sentence_rows.empty:
    print("Warning: empty pattern_sentence values found in these rows:")
    display(empty_sentence_rows)

print("Classifier-level summary:")
display(classifier_summary_df)

expected_classifier_count = pattern_df["classifier"].nunique()
if len(classifier_summary_df) != expected_classifier_count:
    raise ValueError(
        "classifier_summary_df row count does not match detected classifier count: "
        f"{len(classifier_summary_df)} vs {expected_classifier_count}"
    )

sentence_output_path = Path(output_dir) / PATTERN_SENTENCE_FILE
summary_output_path = Path(output_dir) / CLASSIFIER_SUMMARY_FILE
missing_outputs = [
    str(path) for path in [sentence_output_path, summary_output_path] if not path.exists()
]
if missing_outputs:
    raise FileNotFoundError(f"Prompt 6 output files were not saved: {missing_outputs}")

print("Prompt 6 completed: pattern sentences and classifier-level summary generated.")


## Prompt 7 Scenario Summary And LLM-Ready JSON Functions

Input: `pattern_df` with `pattern_sentence`, `classifier_summary_df` with `classifier_summary_sentence`, and Module 1 `metadata`.

Processing: Define JSON-safe conversion helpers and builders for scenario-level and LLM-ready pattern JSON outputs.

Output: Reusable functions for saving `module2_scenario_summary.json` and `module2_llm_ready_patterns.json`.


In [ ]:
# ### Prompt 7 scenario-summary and LLM-ready JSON functions
# These builders produce JSON-safe scenario summaries and final structured objects for the LLM analysis module.

SCENARIO_SUMMARY_FILE = "module2_scenario_summary.json"
LLM_READY_PATTERNS_FILE = "module2_llm_ready_patterns.json"
SCENARIO_SUMMARY_LABEL_COLUMNS = [
    "trend_label",
    "robustness_flag",
    "spike_type",
    "curve_shape",
    "pattern_strength",
    "degradation_type",
]
SCENARIO_COMBINATION_COLUMNS = [
    "classifier",
    "normalization",
    "delta_100_50",
    "error_50",
    "error_100",
    "robustness_flag",
    "degradation_type",
    "pattern_strength",
]
DETECTED_PATTERN_REQUIRED_COLUMNS = [
    "scenario",
    "classifier",
    "normalization",
    "error_50",
    "error_70",
    "error_80",
    "error_90",
    "error_100",
    "delta_100_50",
    "relative_increase_pct",
    "slope",
    "max_jump",
    "max_jump_interval",
    "abs_max_step_change",
    "abs_max_step_interval",
    "trend_label",
    "robustness_flag",
    "spike_flag",
    "spike_type",
    "curve_shape",
    "pattern_strength",
    "degradation_type",
    "pattern_sentence",
]
CLASSIFIER_JSON_REQUIRED_COLUMNS = CLASSIFIER_SUMMARY_COLUMNS


# ### Helper: _to_python_scalar
# Convert pandas/numpy scalar values into JSON-serializable Python scalars.
def _to_python_scalar(value):
    """Convert numpy, pandas, and scalar values to JSON-safe Python values.

    Inputs:
        value: Scalar value that may come from pandas or numpy.

    Output:
        Native Python scalar or None suitable for JSON serialization.
    """
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        value = value.item()
    if isinstance(value, (bool, int, float, str)) or value is None:
        return value
    return str(value)


# ### Helper: _json_safe_record
# Convert a DataFrame row into a JSON-safe dictionary.
def _json_safe_record(record):
    """Recursively convert a record into JSON-serializable Python values.

    Inputs:
        record: Dictionary, list, tuple, or scalar value.

    Output:
        JSON-safe object with native Python values.
    """
    if isinstance(record, dict):
        return {str(key): _json_safe_record(value) for key, value in record.items()}
    if isinstance(record, (list, tuple)):
        return [_json_safe_record(value) for value in record]
    return _to_python_scalar(record)


# ### Helper: _value_counts_dict
# Build a JSON-safe value-count dictionary for a label column.
def _value_counts_dict(series) -> dict[str, int]:
    """Return value counts as a JSON-safe dictionary.

    Inputs:
        series: Pandas Series containing categorical labels.

    Output:
        Dictionary mapping label values to integer counts.
    """
    counts = series.value_counts(dropna=False)
    return {str(key): int(value) for key, value in counts.items()}


# ### Helper: _combination_record
# Format one scenario/classifier/normalization pattern row for JSON output.
def _combination_record(row) -> dict[str, object]:
    """Build one sensitive or stable combination record from a pattern row.

    Inputs:
        row: One row from pattern_df.

    Output:
        JSON-safe dictionary with classifier, normalization, errors, and labels.
    """
    return _json_safe_record({column: row[column] for column in SCENARIO_COMBINATION_COLUMNS})


# ### Function: build_scenario_summary
# Aggregate detected patterns and label distributions at the scenario level.
def build_scenario_summary(pattern_df, metadata: dict, output_dir: str | Path) -> dict[str, object]:
    """Create and save the scenario-level summary JSON.

    Inputs:
        pattern_df: Pattern table with rule-based labels and pattern sentences.
        metadata: Module 1 metadata dictionary.
        output_dir: Directory where module2_scenario_summary.json should be saved.

    Output:
        Scenario-level summary dictionary.
    """
    required_columns = list(set(SCENARIO_SUMMARY_LABEL_COLUMNS + SCENARIO_COMBINATION_COLUMNS + ["scenario"]))
    missing_columns = [column for column in required_columns if column not in pattern_df.columns]
    if missing_columns:
        raise ValueError(f"pattern_df is missing scenario summary columns: {missing_columns}")

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    scenario_values = sorted(pattern_df["scenario"].dropna().astype(str).unique().tolist())
    scenario = scenario_values[0] if scenario_values else None
    split_values = metadata.get("split_values", [50, 70, 80, 90, 100])

    overall_pattern_counts = {
        column: _value_counts_dict(pattern_df[column]) for column in SCENARIO_SUMMARY_LABEL_COLUMNS
    }
    most_sensitive_df = pattern_df.sort_values("delta_100_50", ascending=False).head(5)
    most_stable_df = pattern_df.sort_values("delta_100_50", ascending=True).head(5)

    trend_counts = pattern_df["trend_label"].value_counts()
    increasing_count = int(trend_counts.get("increasing", 0))
    majority_increasing = increasing_count > len(pattern_df) / 2
    if majority_increasing:
        dominant_observation = (
            "Most classifier-normalization curves show increasing error as split moves from 50 to 100, "
            "suggesting reduced robustness under stronger batch-separated evaluation."
        )
    else:
        dominant_observation = "The detected curves show mixed or weak error changes across split levels."

    dominant_degradation_type = _mode_value(pattern_df["degradation_type"])
    dominant_failure_mode = f"The most frequent degradation type is {dominant_degradation_type}."

    scenario_summary = {
        "scenario": scenario,
        "split_values": split_values,
        "metric": "classification error",
        "n_classifiers": int(pattern_df["classifier"].nunique()),
        "n_normalizations": int(pattern_df["normalization"].nunique()),
        "n_curves": int(len(pattern_df)),
        "overall_pattern_counts": overall_pattern_counts,
        "most_sensitive_combinations": [
            _combination_record(row) for _, row in most_sensitive_df.iterrows()
        ],
        "most_stable_combinations": [
            _combination_record(row) for _, row in most_stable_df.iterrows()
        ],
        "dominant_observation": dominant_observation,
        "dominant_failure_mode": dominant_failure_mode,
        "analysis_scope_note": (
            "This summary describes performance patterns only and does not infer biological mechanisms."
        ),
    }
    scenario_summary = _json_safe_record(scenario_summary)

    scenario_summary_path = output_path / SCENARIO_SUMMARY_FILE
    with scenario_summary_path.open("w", encoding="utf-8") as output_file:
        json.dump(scenario_summary, output_file, indent=2)

    return scenario_summary


# ### Function: build_detected_patterns
# Select the most important row-level patterns for inclusion in LLM-ready JSON.
def build_detected_patterns(pattern_df) -> list[dict[str, object]]:
    """Convert curve-level pattern rows into LLM-ready detected pattern records.

    Inputs:
        pattern_df: Pattern table with numeric features, labels, and pattern_sentence.

    Output:
        List of JSON-safe detected pattern dictionaries.
    """
    missing_columns = [column for column in DETECTED_PATTERN_REQUIRED_COLUMNS if column not in pattern_df.columns]
    if missing_columns:
        raise ValueError(f"pattern_df is missing detected pattern columns: {missing_columns}")

    detected_patterns = []
    for _, row in pattern_df.iterrows():
        detected_patterns.append(
            _json_safe_record(
                {
                    "scenario": row["scenario"],
                    "classifier": row["classifier"],
                    "normalization": row["normalization"],
                    "error_curve": {
                        "50": row["error_50"],
                        "70": row["error_70"],
                        "80": row["error_80"],
                        "90": row["error_90"],
                        "100": row["error_100"],
                    },
                    "delta_100_50": row["delta_100_50"],
                    "relative_increase_pct": row["relative_increase_pct"],
                    "slope": row["slope"],
                    "max_jump": row["max_jump"],
                    "max_jump_interval": row["max_jump_interval"],
                    "abs_max_step_change": row["abs_max_step_change"],
                    "abs_max_step_interval": row["abs_max_step_interval"],
                    "trend_label": row["trend_label"],
                    "robustness_flag": row["robustness_flag"],
                    "spike_flag": _as_bool(row["spike_flag"]),
                    "spike_type": row["spike_type"],
                    "curve_shape": row["curve_shape"],
                    "pattern_strength": row["pattern_strength"],
                    "degradation_type": row["degradation_type"],
                    "pattern_sentence": row["pattern_sentence"],
                }
            )
        )
    return detected_patterns


# ### Function: build_classifier_level_summary_records
# Convert classifier summaries into JSON-safe records.
def build_classifier_level_summary_records(classifier_summary_df) -> list[dict[str, object]]:
    """Convert classifier-level summary rows into JSON-safe records.

    Inputs:
        classifier_summary_df: Classifier summary table from Prompt 6.

    Output:
        List of JSON-safe classifier-level summary dictionaries.
    """
    missing_columns = [column for column in CLASSIFIER_JSON_REQUIRED_COLUMNS if column not in classifier_summary_df.columns]
    if missing_columns:
        raise ValueError(f"classifier_summary_df is missing JSON columns: {missing_columns}")
    return [
        _json_safe_record({column: row[column] for column in CLASSIFIER_JSON_REQUIRED_COLUMNS})
        for _, row in classifier_summary_df.iterrows()
    ]


# ### Function: build_llm_ready_patterns_json
# Build and save the final structured JSON consumed by downstream LLM analysis.
def build_llm_ready_patterns_json(
    pattern_df,
    classifier_summary_df,
    metadata: dict,
    scenario_summary: dict,
    output_dir: str | Path,
) -> dict[str, object]:
    """Create and save the final LLM-ready pattern JSON object.

    Inputs:
        pattern_df: Pattern table with pattern_sentence.
        classifier_summary_df: Classifier summary table with classifier_summary_sentence.
        metadata: Module 1 metadata dictionary.
        scenario_summary: Scenario-level summary dictionary.
        output_dir: Directory where module2_llm_ready_patterns.json should be saved.

    Output:
        JSON-safe LLM-ready pattern dictionary.
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    scenario_values = sorted(pattern_df["scenario"].dropna().astype(str).unique().tolist())
    scenario = scenario_values[0] if scenario_values else None

    llm_ready_patterns = {
        "module": "Module 2: Pattern Detection Engine",
        "input_from_module1": {
            "performance_summary_file": "module1_performance_summary.csv",
            "metadata_file": "module1_metadata_summary.json",
        },
        "experiment_context": {
            "scenario": scenario,
            "classifiers": sorted(pattern_df["classifier"].dropna().astype(str).unique().tolist()),
            "normalizations": sorted(pattern_df["normalization"].dropna().astype(str).unique().tolist()),
            "split_values": metadata.get("split_values", [50, 70, 80, 90, 100]),
            "metric": "classification error",
            "interpretation_note": "Higher split values indicate stronger batch-separated evaluation.",
            "analysis_scope_note": (
                "Use only the detected pattern labels and numeric features. "
                "Do not infer biological mechanisms."
            ),
        },
        "detected_patterns": build_detected_patterns(pattern_df),
        "classifier_level_summary": build_classifier_level_summary_records(classifier_summary_df),
        "scenario_level_summary": scenario_summary,
    }
    llm_ready_patterns = _json_safe_record(llm_ready_patterns)

    llm_ready_path = output_path / LLM_READY_PATTERNS_FILE
    with llm_ready_path.open("w", encoding="utf-8") as output_file:
        json.dump(llm_ready_patterns, output_file, indent=2)

    return llm_ready_patterns


## Run Prompt 7 Scenario Summary And LLM-Ready JSON Generation

Input: `pattern_df`, `classifier_summary_df`, and Module 1 `metadata`.

Processing: Save scenario-level summary JSON, save final LLM-ready pattern JSON, reload both files, and validate counts.

Output: `scenario_summary`, `llm_ready_patterns`, saved JSON files, previews, and Prompt 7 completion message.


In [ ]:
# ### Execute Prompt 7
# Save and reload scenario-level and LLM-ready JSON outputs, then validate their counts.

scenario_summary = build_scenario_summary(pattern_df, metadata, output_dir)
llm_ready_patterns = build_llm_ready_patterns_json(
    pattern_df,
    classifier_summary_df,
    metadata,
    scenario_summary,
    output_dir,
)

scenario_summary_path = Path(output_dir) / SCENARIO_SUMMARY_FILE
llm_ready_path = Path(output_dir) / LLM_READY_PATTERNS_FILE

with scenario_summary_path.open("r", encoding="utf-8") as input_file:
    loaded_scenario_summary = json.load(input_file)
with llm_ready_path.open("r", encoding="utf-8") as input_file:
    loaded_llm_ready_patterns = json.load(input_file)

print("Scenario summary top-level keys:")
print(list(loaded_scenario_summary.keys()))
print("LLM-ready JSON top-level keys:")
print(list(loaded_llm_ready_patterns.keys()))

print("First two detected patterns:")
display(loaded_llm_ready_patterns["detected_patterns"][:2])

print("Scenario-level summary:")
display(loaded_llm_ready_patterns["scenario_level_summary"])

if len(loaded_llm_ready_patterns["detected_patterns"]) != len(pattern_df):
    raise ValueError(
        "detected_patterns count does not match pattern_df row count: "
        f"{len(loaded_llm_ready_patterns['detected_patterns'])} vs {len(pattern_df)}"
    )

if len(loaded_llm_ready_patterns["classifier_level_summary"]) != len(classifier_summary_df):
    raise ValueError(
        "classifier_level_summary count does not match classifier_summary_df row count: "
        f"{len(loaded_llm_ready_patterns['classifier_level_summary'])} vs {len(classifier_summary_df)}"
    )

print(
    "Prompt 7 completed: LLM-ready JSON generated. "
    "Main output: module2_outputs/module2_llm_ready_patterns.json"
)


## Prompt 8 Visualization And Final Checklist Functions

Input: `pattern_df` from Prompt 7.

Processing: Define matplotlib-only plotting helpers and final Module 2 output checklist validation.

Output: Reusable functions that save Module 2 figures and print the final output checklist.


In [ ]:
# ### Prompt 8 visualization and checklist functions
# These plotting and checklist helpers create demo/report figures and verify that all Module 2 deliverables exist.

try:
    import matplotlib.pyplot as plt
except Exception:
    try:
        matplotlib = ensure_package("matplotlib")
        import matplotlib.pyplot as plt
    except Exception:
        print("Retrying matplotlib install without upgrading existing local dependencies.")
        LOCAL_DEPS.mkdir(exist_ok=True)
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--target",
                str(LOCAL_DEPS),
                "--no-deps",
                "matplotlib",
            ]
        )
        sys.path.insert(0, str(LOCAL_DEPS))
        importlib.invalidate_caches()
        sys.modules.pop("matplotlib", None)
        import matplotlib.pyplot as plt

FIGURES_DIR_NAME = "figures"
ERROR_SPLITS = [50, 70, 80, 90, 100]
ERROR_CURVE_COLUMNS = ["error_50", "error_70", "error_80", "error_90", "error_100"]
FINAL_REQUIRED_OUTPUTS = [
    "module2_error_curve_table.csv",
    "module2_numeric_features.csv",
    "module2_pattern_table.csv",
    "module2_pattern_table_with_sentences.csv",
    "module2_classifier_summary.csv",
    "module2_scenario_summary.json",
    "module2_llm_ready_patterns.json",
]
FINAL_REQUIRED_FIGURES = [
    "delta_100_50_heatmap.png",
    "robustness_flag_counts.png",
    "degradation_type_counts.png",
]


# ### Helper: _figures_dir
# Create and return the Module 2 figures directory.
def _figures_dir(output_dir: str | Path) -> Path:
    """Return and create the Module 2 figures directory.

    Inputs:
        output_dir: Module 2 output directory.

    Output:
        Path to module2_outputs/figures.
    """
    figures_dir = Path(output_dir) / FIGURES_DIR_NAME
    figures_dir.mkdir(parents=True, exist_ok=True)
    return figures_dir


# ### Function: plot_classifier_error_curves
# Save line plots showing error curves by classifier and normalization.
def plot_classifier_error_curves(pattern_df, output_dir: str | Path) -> list[Path]:
    """Create one error-curve line plot per classifier.

    Inputs:
        pattern_df: Pattern DataFrame containing error_50 through error_100 columns.
        output_dir: Module 2 output directory where figures should be saved.

    Output:
        List of saved classifier error-curve PNG paths.
    """
    missing_columns = [column for column in ["scenario", "classifier", "normalization"] + ERROR_CURVE_COLUMNS if column not in pattern_df.columns]
    if missing_columns:
        raise ValueError(f"pattern_df is missing error-curve plot columns: {missing_columns}")

    figures_dir = _figures_dir(output_dir)
    saved_paths = []
    for classifier in sorted(pattern_df["classifier"].dropna().astype(str).unique()):
        classifier_df = pattern_df[pattern_df["classifier"].astype(str) == classifier].copy()
        scenario_values = sorted(classifier_df["scenario"].dropna().astype(str).unique().tolist())
        scenario = scenario_values[0] if scenario_values else "unknown_scenario"

        fig, ax = plt.subplots(figsize=(8, 5))
        max_error = 0.0
        for _, row in classifier_df.sort_values("normalization").iterrows():
            errors = [float(row[column]) for column in ERROR_CURVE_COLUMNS]
            max_error = max(max_error, max(errors))
            ax.plot(ERROR_SPLITS, errors, marker="o", linewidth=2, label=row["normalization"])

        y_max = max_error * 1.10 if max_error > 0 else 1.0
        ax.set_ylim(bottom=0, top=y_max)
        ax.set_xlabel("Split")
        ax.set_ylabel("Classification error")
        ax.set_title(f"Error Curves for {scenario} - {classifier}")
        ax.set_xticks(ERROR_SPLITS)
        ax.grid(True, alpha=0.3)
        ax.legend(title="Normalization", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
        fig.tight_layout()

        figure_path = figures_dir / f"error_curve_{classifier}.png"
        fig.savefig(figure_path, dpi=150, bbox_inches="tight")
        display(fig)
        plt.close(fig)
        saved_paths.append(figure_path)

    return saved_paths


# ### Function: plot_delta_heatmap
# Save a heatmap-like table figure of cross-batch degradation values.
def plot_delta_heatmap(pattern_df, output_dir: str | Path) -> Path:
    """Create a classifier by normalization heatmap for delta_100_50.

    Inputs:
        pattern_df: Pattern DataFrame containing classifier, normalization, and delta_100_50.
        output_dir: Module 2 output directory where the heatmap should be saved.

    Output:
        Saved delta heatmap PNG path.
    """
    missing_columns = [column for column in ["classifier", "normalization", "delta_100_50"] if column not in pattern_df.columns]
    if missing_columns:
        raise ValueError(f"pattern_df is missing heatmap columns: {missing_columns}")

    figures_dir = _figures_dir(output_dir)
    heatmap_df = pattern_df.pivot(index="classifier", columns="normalization", values="delta_100_50")
    heatmap_df = heatmap_df.sort_index().reindex(sorted(heatmap_df.columns), axis=1)

    fig, ax = plt.subplots(figsize=(8, 5))
    image = ax.imshow(heatmap_df.values.astype(float), aspect="auto", cmap="viridis")
    ax.set_title("Delta Error from Split 50 to 100")
    ax.set_xlabel("Normalization")
    ax.set_ylabel("Classifier")
    ax.set_xticks(range(len(heatmap_df.columns)))
    ax.set_xticklabels(heatmap_df.columns)
    ax.set_yticks(range(len(heatmap_df.index)))
    ax.set_yticklabels(heatmap_df.index)

    for row_index, classifier in enumerate(heatmap_df.index):
        for col_index, normalization in enumerate(heatmap_df.columns):
            value = heatmap_df.loc[classifier, normalization]
            label = "NA" if pd.isna(value) else f"{value:.3f}"
            ax.text(col_index, row_index, label, ha="center", va="center", color="white")

    fig.colorbar(image, ax=ax, label="delta_100_50")
    fig.tight_layout()

    figure_path = figures_dir / "delta_100_50_heatmap.png"
    fig.savefig(figure_path, dpi=150, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    return figure_path


# ### Function: plot_label_count_bar
# Save count plots for selected rule-based labels.
def plot_label_count_bar(
    pattern_df,
    label_column: str,
    title: str,
    output_name: str,
    output_dir: str | Path,
    rotate_labels: bool = False,
) -> Path:
    """Create and save a count bar plot for one label column.

    Inputs:
        pattern_df: Pattern DataFrame containing the requested label column.
        label_column: Column whose values should be counted.
        title: Plot title.
        output_name: PNG filename to save in module2_outputs/figures.
        output_dir: Module 2 output directory.
        rotate_labels: Whether to rotate x-axis labels by 30 degrees.

    Output:
        Saved bar plot PNG path.
    """
    if label_column not in pattern_df.columns:
        raise ValueError(f"pattern_df is missing label column: {label_column}")

    figures_dir = _figures_dir(output_dir)
    counts = pattern_df[label_column].value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(counts.index.astype(str), counts.values)
    ax.set_title(title)
    ax.set_xlabel(label_column)
    ax.set_ylabel("Count")
    if rotate_labels:
        ax.tick_params(axis="x", labelrotation=30)
        for label in ax.get_xticklabels():
            label.set_horizontalalignment("right")

    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height, str(int(height)), ha="center", va="bottom")

    fig.tight_layout()
    figure_path = figures_dir / output_name
    fig.savefig(figure_path, dpi=150, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    return figure_path


# ### Function: generate_module2_visualizations
# Run all visualization helpers and return saved figure paths.
def generate_module2_visualizations(pattern_df, output_dir: str | Path) -> list[Path]:
    """Generate all Module 2 visualization figures.

    Inputs:
        pattern_df: Pattern DataFrame from Prompt 7.
        output_dir: Module 2 output directory.

    Output:
        List of saved figure paths.
    """
    saved_paths = []
    saved_paths.extend(plot_classifier_error_curves(pattern_df, output_dir))
    saved_paths.append(plot_delta_heatmap(pattern_df, output_dir))
    saved_paths.append(
        plot_label_count_bar(
            pattern_df,
            "robustness_flag",
            "Robustness Flag Distribution",
            "robustness_flag_counts.png",
            output_dir,
        )
    )
    saved_paths.append(
        plot_label_count_bar(
            pattern_df,
            "degradation_type",
            "Degradation Type Distribution",
            "degradation_type_counts.png",
            output_dir,
            rotate_labels=True,
        )
    )
    return saved_paths


# ### Function: print_module2_final_checklist
# Print whether every expected Module 2 output file has been generated.
def print_module2_final_checklist(output_dir: str | Path) -> None:
    """Print final Module 2 output paths and warn about missing files.

    Inputs:
        output_dir: Module 2 output directory.

    Output:
        None. Prints checklist and success or warning messages.
    """
    output_path = Path(output_dir)
    figures_dir = output_path / FIGURES_DIR_NAME
    missing_paths = []

    print("Module 2 final output checklist:")
    for filename in FINAL_REQUIRED_OUTPUTS:
        path = output_path / filename
        print(path)
        if not path.exists():
            missing_paths.append(path)

    error_curve_paths = sorted(figures_dir.glob("error_curve_*.png")) if figures_dir.exists() else []
    for path in error_curve_paths:
        print(path)

    for filename in FINAL_REQUIRED_FIGURES:
        path = figures_dir / filename
        print(path)
        if not path.exists():
            missing_paths.append(path)

    figure_paths = sorted(figures_dir.glob("*.png")) if figures_dir.exists() else []
    if len(figure_paths) < 9:
        print(f"Warning: expected at least 9 figures, found {len(figure_paths)}.")

    expected_curve_count = 6
    if len(error_curve_paths) < expected_curve_count:
        print(f"Warning: expected at least {expected_curve_count} classifier error curve figures, found {len(error_curve_paths)}.")

    if missing_paths:
        print("Warning: missing required Module 2 outputs:")
        for path in missing_paths:
            print(path)
    elif len(figure_paths) >= 9 and len(error_curve_paths) >= expected_curve_count:
        print(
            "Module 2 completed successfully. The main output for Module 3 is "
            "module2_outputs/module2_llm_ready_patterns.json."
        )


## Run Prompt 8 Visualizations And Final Checklist

Input: `pattern_df` from Prompt 7.

Processing: Generate all required matplotlib figures and print the final Module 2 output checklist.

Output: Saved PNG figures, checklist output, and Prompt 8 completion message.


In [ ]:
# ### Execute Prompt 8
# Generate figures and print the final Module 2 output checklist.

figure_paths = generate_module2_visualizations(pattern_df, output_dir)
print("Saved Module 2 figure paths:")
for figure_path in figure_paths:
    print(figure_path)

print_module2_final_checklist(output_dir)

print("Prompt 8 completed: Module 2 visualizations and final checklist generated.")
